In [34]:
# CELL 0: Setup & imports
import os, time, json, glob, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
import joblib
from IPython.display import Audio, display, Javascript

# Data path (the file you uploaded)
data_path = "sbpf_fault_data_full.csv"
print("Data path:", data_path)


Data path: sbpf_fault_data_full.csv


In [35]:
# CELL 1: Mount Drive and create save folder (run once)
from google.colab import drive
drive.mount('/content/drive')   # follow auth link

SAVE_DIR = "/content/drive/MyDrive/SKBPF_Results"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Results will be saved to:", SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results will be saved to: /content/drive/MyDrive/SKBPF_Results


In [36]:
# CELL 2: Load data and split (70 / 15 / 15)
df = pd.read_csv(data_path)
print("CSV loaded:", df.shape)
# label column name in your file
label_col = "Fault_Class"
if label_col not in df.columns:
    raise RuntimeError(f"Label column '{label_col}' not found. Columns: {df.columns.tolist()}")

X = df.drop(columns=[label_col])
y = df[label_col].astype(str)

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
class_names = le.classes_
print("Classes:", list(class_names))

# Splits: Train 70%, Val 15%, Test 15% (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(X, y_enc, test_size=0.30, stratify=y_enc, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


CSV loaded: (5500, 801)
Classes: ['F0', 'F1', 'F10', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9']
Train: (3850, 800) Val: (825, 800) Test: (825, 800)


In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

CV_FOLDS = 5  # optimized for speed

def save_to_drive(src_filename):
    dst = os.path.join(SAVE_DIR, src_filename)
    shutil.copy(src_filename, dst)
    print("Copied to Drive:", dst)

def evaluate_and_save(name, pipeline, X_local, y_local, classes, n_jobs=-1):
    print(f"\n--- Running {name} ---")
    start = time.time()
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    # CV accuracy
    scores = cross_val_score(pipeline, X_local, y_local, cv=cv, scoring='accuracy', n_jobs=n_jobs)
    mean_acc, std_acc = float(scores.mean()), float(scores.std())
    # CV predictions aggregated for confusion matrix
    y_pred = cross_val_predict(pipeline, X_local, y_local, cv=cv, n_jobs=n_jobs)
    cm = confusion_matrix(y_local, y_pred).tolist()
    report = classification_report(y_local, y_pred, target_names=classes, zero_division=0)

    result = {
        "model": name,
        "cv_folds": CV_FOLDS,
        "cv_mean_accuracy": mean_acc,
        "cv_std_accuracy": std_acc,
        "classification_report": report,
        "confusion_matrix": cm,
        "timestamp": time.strftime("%Y%m%d-%H%M%S", time.gmtime())
    }
    filename = f"{name}_results.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent=2)
    # copy to Drive
    save_to_drive(filename)
    elapsed = time.time() - start
    print(f"{name} done. CV acc={mean_acc:.4f} \u00b1 {std_acc:.4f}. Time {elapsed:.1f}s")
    return result

In [10]:
# CELL 4: Fine KNN (n=1)
pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))
res_finek = evaluate_and_save("FineKNN", pipeline, X_train, y_train, class_names, n_jobs=-1)



--- Running FineKNN ---
Copied to Drive: /content/drive/MyDrive/SKBPF_Results/FineKNN_results.json
FineKNN done. CV acc=0.9894 ± 0.0059. Time 2.7s


In [23]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

CV_FOLDS = 5  # optimized for speed

def save_to_drive(src_filename):
    dst = os.path.join(SAVE_DIR, src_filename)
    shutil.copy(src_filename, dst)
    print("Copied to Drive:", dst)

def evaluate_and_save(name, pipeline, X_local, y_local, classes, n_jobs=-1):
    print(f"\n--- Running {name} ---")
    start = time.time()
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    # CV accuracy
    scores = cross_val_score(pipeline, X_local, y_local, cv=cv, scoring='accuracy', n_jobs=n_jobs)
    mean_acc, std_acc = float(scores.mean()), float(scores.std())
    # CV predictions aggregated for confusion matrix
    y_pred = cross_val_predict(pipeline, X_local, y_local, cv=cv, n_jobs=n_jobs)
    cm = confusion_matrix(y_local, y_pred).tolist()
    report = classification_report(y_local, y_pred, target_names=classes, zero_division=0)

    result = {
        "model": name,
        "cv_folds": CV_FOLDS,
        "cv_mean_accuracy": mean_acc,
        "cv_std_accuracy": std_acc,
        "classification_report": report,
        "confusion_matrix": cm,
        "timestamp": time.strftime("%Y%m%d-%H%M%S", time.gmtime())
    }
    filename = f"{name}_results.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent=2)
    # copy to Drive
    save_to_drive(filename)
    elapsed = time.time() - start
    print(f"{name} done. CV acc={mean_acc:.4f} \u00b1 {std_acc:.4f}. Time {elapsed:.1f}s")
    return result

In [24]:
# CELL 4: Fine KNN (n=1)
pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))
res_finek = evaluate_and_save("FineKNN", pipeline, X_train, y_train, class_names, n_jobs=-1)


--- Running FineKNN ---
Copied to Drive: /content/drive/MyDrive/SKBPF_Results/FineKNN_results.json
FineKNN done. CV acc=0.9894 ± 0.0059. Time 2.3s


In [25]:
# CELL 5: Med KNN (n=5)
pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
res_medk = evaluate_and_save("MedKNN", pipeline, X_train, y_train, class_names, n_jobs=-1)


--- Running MedKNN ---
Copied to Drive: /content/drive/MyDrive/SKBPF_Results/MedKNN_results.json
MedKNN done. CV acc=0.9403 ± 0.0104. Time 3.4s


In [13]:
# CELL 6a: Weighted KNN
pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=10, weights='distance'))
res_knn_w = evaluate_and_save("KNN_weighted", pipeline, X_train, y_train, class_names, n_jobs=-1)



--- Running KNN_weighted ---
Copied to Drive: /content/drive/MyDrive/SKBPF_Results/KNN_weighted_results.json
KNN_weighted done. CV acc=0.9582 ± 0.0105. Time 3.3s


In [14]:
# CELL 6b: Cosine KNN
pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, metric='cosine'))
res_cosine = evaluate_and_save("CosineKNN", pipeline, X_train, y_train, class_names, n_jobs=-1)



--- Running CosineKNN ---
Copied to Drive: /content/drive/MyDrive/SKBPF_Results/CosineKNN_results.json
CosineKNN done. CV acc=0.9509 ± 0.0092. Time 5.3s


In [15]:
# CELL 6c: Coarse KNN
pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=20))
res_coarse = evaluate_and_save("CoarseKNN", pipeline, X_train, y_train, class_names, n_jobs=-1)



--- Running CoarseKNN ---
Copied to Drive: /content/drive/MyDrive/SKBPF_Results/CoarseKNN_results.json
CoarseKNN done. CV acc=0.7519 ± 0.0189. Time 3.2s


In [16]:
# CELL 7: RandomForest (BaggedTrees approx) - n_estimators reduced for speed
pipeline = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=60, n_jobs=-1, random_state=42))
res_rf = evaluate_and_save("BaggedTrees_RF", pipeline, X_train, y_train, class_names, n_jobs=-1)



--- Running BaggedTrees_RF ---
Copied to Drive: /content/drive/MyDrive/SKBPF_Results/BaggedTrees_RF_results.json
BaggedTrees_RF done. CV acc=0.9603 ± 0.0112. Time 315.5s


In [19]:
# CELL 9: SVM cubic (moderate)
pipeline = make_pipeline(StandardScaler(), SVC(kernel='poly', degree=3, C=1.0))
res_svm = evaluate_and_save("SVM_cubic", pipeline, X_train, y_train, class_names, n_jobs=-1)



--- Running SVM_cubic ---


KeyboardInterrupt: 

In [18]:
# CELL 10: NN_small - early stopping, limited iter
pipeline = make_pipeline(StandardScaler(),
                         MLPClassifier(hidden_layer_sizes=(40,), max_iter=300,
                                       early_stopping=True, n_iter_no_change=10, random_state=42))
res_nn_small = evaluate_and_save("NN_small", pipeline, X_train, y_train, class_names, n_jobs=-1)



--- Running NN_small ---
Copied to Drive: /content/drive/MyDrive/SKBPF_Results/NN_small_results.json
NN_small done. CV acc=0.7481 ± 0.0599. Time 25.2s


In [20]:
# CELL 11: gather all result JSONs saved in working dir / Drive and pick top 2 by cv_mean_accuracy
import glob, json, operator
json_files = glob.glob("*_results.json")
summary = []
for jf in json_files:
    with open(jf) as f:
        r = json.load(f)
        summary.append( (jf.replace("_results.json",""), r["cv_mean_accuracy"], r["cv_std_accuracy"]) )
summary_df = pd.DataFrame(summary, columns=["model","cv_mean_acc","cv_std"])
summary_df = summary_df.sort_values(by="cv_mean_acc", ascending=False).reset_index(drop=True)
display(summary_df)
summary_df.to_csv("models_summary.csv", index=False)
shutil.copy("models_summary.csv", os.path.join(SAVE_DIR, "models_summary.csv"))
print("Saved summary to Drive.")
# pick top 2
top2 = summary_df['model'].tolist()[:2]
print("Top 2 models:", top2)


,model,cv_mean_acc,cv_std
0,FineKNN,0.989351,0.005889
1,BaggedTrees_RF,0.960260,0.011160
2,KNN_weighted,0.958182,0.010499
3,CosineKNN,0.950909,0.009198
4,MedKNN,0.940260,0.010422
5,CoarseKNN,0.751948,0.018945
6,NN_small,0.748052,0.059949
7,SVM_cubic,0.294286,0.024677


Saved summary to Drive.
Top 2 models: ['FineKNN', 'BaggedTrees_RF']


In [40]:
# === FINAL TOP-2 TUNING (SKIPS BAGGED TREES) ===
# Bagged Trees is slow → we skip tuning and use default.
# Only fast models (KNN, AdaBoost) are tuned.

import joblib

print("\n=== FINAL FAST TUNING FOR TOP 2 MODELS ===")

best_models = {}

FAST_MODELS = [
    "FineKNN", "MedKNN", "KNN_weighted", "CosineKNN", "CoarseKNN",
    "AdaBoost_DT"
]

fast_spaces = {
    "KNeighborsClassifier": {
        "n_neighbors": [3, 5],
        "weights": ["uniform", "distance"]
    },
    "AdaBoostClassifier": {
        "n_estimators": [40, 60]
    }
}

def fast_tune(model_name):
    est = model_map[model_name]
    key = est.__class__.__name__

    # If it's not a fast model, skip tuning
    if model_name not in FAST_MODELS:
        print(f"\nSkipping tuning for {model_name} (SLOW model). Using default version.")
        pipe = make_pipeline(StandardScaler(), est)
        pipe.fit(X_train, y_train)
        return pipe

    # Build pipeline
    pipe = make_pipeline(StandardScaler(), est)

    # Build small param grid
    param_grid = {
        f"{est.__class__.__name__.lower()}__{p}": v
        for p, v in fast_spaces[key].items()
    }

    print(f"\nTuning FAST model {model_name}...")
    grid = GridSearchCV(
        pipe, param_grid,
        cv=3,            # fast
        n_jobs=-1,
        scoring="accuracy"
    )
    grid.fit(X_train, y_train)

    print("Best parameters:", grid.best_params_)
    print("Best CV score:", grid.best_score_)
    return grid.best_estimator_


# run for top2
for model_name in top2:
    print("\nProcessing:", model_name)
    tuned_model = fast_tune(model_name)
    best_models[model_name] = tuned_model

    # save model
    filename = f"{model_name}_FASTtuned.joblib"
    joblib.dump(tuned_model, filename)
    shutil.copy(filename, os.path.join(SAVE_DIR, filename))

    print(f"Saved {model_name} to Drive.")


print("\n=== FAST TUNING COMPLETE ===")



=== FINAL FAST TUNING FOR TOP 2 MODELS ===

Processing: FineKNN

Tuning FAST model FineKNN...
Best parameters: {'kneighborsclassifier__n_neighbors': 3, 'kneighborsclassifier__weights': 'distance'}
Best CV score: 0.9776638184939407
Saved FineKNN to Drive.

Processing: BaggedTrees_RF

Skipping tuning for BaggedTrees_RF (SLOW model). Using default version.
Saved BaggedTrees_RF to Drive.

=== FAST TUNING COMPLETE ===


In [42]:
# CELL 13: Train chosen best model(s) on Train+Val and evaluate on Test
# We'll pick the top tuned model (highest best CV if we did tuning), else top1 from summary

# pick a final model pipeline:
if best_models:
    chosen_name = list(best_models.keys())[0]
    final_pipeline = best_models[chosen_name]
    print("Chosen (from tuning):", chosen_name)
else:
    chosen_name = summary_df.loc[0,"model"]
    # load the saved pipeline (if present) or construct pipeline from mapping
    print("Chosen (from summary):", chosen_name)
    # fallback: use simple saved pipeline
    try:
        final_pipeline = joblib.load(f"{chosen_name}_best_pipeline.joblib")
    except:
        # fallback to re-create default pipeline quickly
        default_est = model_map.get(chosen_name, KNeighborsClassifier(n_neighbors=1))
        final_pipeline = make_pipeline(StandardScaler(), default_est)

# combine train+val
X_final_train = pd.concat([X_train, X_val])
y_final_train = np.concatenate([y_train, y_val])
print("Training final pipeline on train+val...")
final_pipeline.fit(X_final_train, y_final_train)

# evaluate on test set
y_test_pred = final_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_test_pred)
print("Final test accuracy:", acc)
print(classification_report(y_test, y_test_pred, target_names=class_names))
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion matrix:\n", cm)

# save final pipeline
joblib.dump(final_pipeline, f"FINAL_{chosen_name}_pipeline.joblib")
shutil.copy(f"FINAL_{chosen_name}_pipeline.joblib", os.path.join(SAVE_DIR, f"FINAL_{chosen_name}_pipeline.joblib"))
print("Saved final model to Drive.")

Chosen (from tuning): FineKNN
Training final pipeline on train+val...
Final test accuracy: 1.0
              precision    recall  f1-score   support

          F0       1.00      1.00      1.00        75
          F1       1.00      1.00      1.00        75
         F10       1.00      1.00      1.00        75
          F2       1.00      1.00      1.00        75
          F3       1.00      1.00      1.00        75
          F4       1.00      1.00      1.00        75
          F5       1.00      1.00      1.00        75
          F6       1.00      1.00      1.00        75
          F7       1.00      1.00      1.00        75
          F8       1.00      1.00      1.00        75
          F9       1.00      1.00      1.00        75

    accuracy                           1.00       825
   macro avg       1.00      1.00      1.00       825
weighted avg       1.00      1.00      1.00       825

Confusion matrix:
 [[75  0  0  0  0  0  0  0  0  0  0]
 [ 0 75  0  0  0  0  0  0  0  0  0]
 